# vector-normalize-keepdim — ex1: row-wise L2 normalize a batch of embeddings

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `vector-normalize-keepdim`. Running the final beacon cell reports progress against the `PyTorch: vector normalize keepdim` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: vector normalize keepdim` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`vector-normalize-keepdim`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "vector-normalize-keepdim"
DD_SUBTOPIC = "PyTorch: vector normalize keepdim"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## vector normalize with keepdim — quick refresher

Unit-normalizing along the last axis is `x / x.norm(dim=-1, keepdim=True)`. The `keepdim=True` is the load-bearing part: without it the norm tensor loses the last axis (`(B,)` instead of `(B, 1)`) and the divide either fails or broadcasts the wrong way.

**Why keepdim.** Broadcast rules align trailing dims. If `x` is `(B, D)` and the divisor is `(B,)`, broadcasting tries to align `(B,)` with the LAST axis of `x` (size `D`) — wrong. With `keepdim=True` the divisor is `(B, 1)`, which aligns with the trailing `D` and broadcasts across it.

Use it for cosine-similarity inputs, contrastive embeddings, RoPE rotation arrays, or any 'put each row on the unit sphere' move.

### Exercise 1 — row-wise L2 normalize a batch of embeddings

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `x.norm(dim=-1, keepdim=True)` to L2-normalize every row of a batch of embeddings so each row lies on the unit sphere.
> Keywords: normalize, keepdim, broadcasting, embeddings
> ```

**KCs targeted:** `norm-with-keepdim`, `broadcast-divide`

Implement `ex1_row_normalize(x)`. Given an embedding batch shaped `(B, D)`:

1. Compute the per-row L2 norm with `x.norm(dim=-1, keepdim=True)` → shape `(B, 1)`.
2. Divide `x` by that norm; broadcasting expands `(B, 1)` over the `D` trailing dimension.
3. Return the normalized tensor, same shape as `x`.

Edge case: if a row is the zero vector, `0 / 0` produces `nan`. The test does NOT pass zero rows — but be aware that real pipelines add a small `eps` (e.g. `x.norm(...).clamp(min=1e-12)`) to handle the case.

Input: `(B, D)` float tensor.
Output: `(B, D)` float tensor; every row has L2 norm `≈ 1.0`.

In [ ]:
def ex1_row_normalize(x: Tensor) -> Tensor:
    """L2-normalize each row of x with keepdim broadcasting."""
    raise NotImplementedError()


def _test_ex1():
    x = t.tensor([
        [3.0, 4.0],         # norm 5, normalized = (0.6, 0.8)
        [0.0, 1.0],         # already unit
        [-1.0, 0.0],        # already unit
        [2.0, 2.0, 1.0],    # different D — handled separately below
    ][:3])
    out = ex1_row_normalize(x)
    assert out.shape == x.shape
    expected = t.tensor([
        [0.6, 0.8],
        [0.0, 1.0],
        [-1.0, 0.0],
    ])
    assert t.allclose(out, expected, atol=1e-6), f'got {out}'
    # Each row has unit norm.
    row_norms = out.norm(dim=-1)
    assert t.allclose(row_norms, t.ones(3), atol=1e-6), f'row norms {row_norms}'

    # Larger random batch — every row must end with L2 norm ~1.
    rng = t.Generator().manual_seed(2)
    big = t.randn(64, 128, generator=rng)
    big_out = ex1_row_normalize(big)
    assert big_out.shape == (64, 128)
    assert t.allclose(big_out.norm(dim=-1), t.ones(64), atol=1e-5)

    # Cosine similarity between row 0 and itself must be 1.0 after normalize.
    v = t.tensor([[1.0, 2.0, 3.0, 4.0]])
    v_n = ex1_row_normalize(v)
    cos = (v_n * v_n).sum(dim=-1)
    assert t.allclose(cos, t.tensor([1.0]), atol=1e-6)
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_row_normalize(x: Tensor) -> Tensor:
    return x / x.norm(dim=-1, keepdim=True)
```

**`keepdim=True` is the only interesting line.** Drop it and you get a shape error (or worse, a silent wrong-axis broadcast). The rule: when you want the reduction result to **broadcast back over the reduced axis**, keep the dim.

**Why not `F.normalize`.** `torch.nn.functional.normalize(x, dim=-1)` does exactly this AND adds an `eps` for numerical safety. In production prefer it. The manual version is the drill because it makes the `keepdim` mechanic visible.

**Cosine-similarity setup.** After row-normalize, `a @ b.transpose(0, 1)` gives the full pairwise cosine matrix. This is the entire setup for contrastive losses (CLIP, SimCLR).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()